In [ ]:
import pandas as pd
df = pd.read_csv('/content/dataset.csv')
df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Smart Fertilizer Recommendation System (Using Real Dataset)

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# -------------------------------
# Step 1: Load Dataset
# -------------------------------

df = pd.read_csv("dataset.csv") # Corrected filename

# Fix column names (remove spaces and replace with underscores)
df.columns = df.columns.str.strip().str.replace(' ', '_')

# -------------------------------
# Step 2: Encode Categorical Data
# -------------------------------

le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fertilizer = LabelEncoder()

df['Soil_Type'] = le_soil.fit_transform(df['Soil_Type'])
df['Crop_Type'] = le_crop.fit_transform(df['Crop_Type'])
df['Fertilizer_Name'] = le_fertilizer.fit_transform(df['Fertilizer_Name'])

# -------------------------------
# Step 3: Define Features & Target
# -------------------------------

X = df[['Nitrogen', 'Phosphorous', 'Potassium', 'Temparature', 'Moisture',
        'Soil_Type', 'Crop_Type']]

y = df['Fertilizer_Name']

# -------------------------------
# Step 4: Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# -------------------------------
# Step 5: Train Model
# -------------------------------

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# -------------------------------
# Step 6: Evaluate Model
# -------------------------------

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy * 100, "% ")

# -------------------------------
# Step 7: Prediction (User Input)
# -------------------------------

# Example input (you can change in viva)
sample_data = [[30, 60, 40,   # N, P, K
                25, 60,         # temperature, moisture
                1, 2]]  # Soil Type, Crop Type (encoded values)

# Convert sample to DataFrame with correct column names to avoid UserWarning
sample_df = pd.DataFrame(sample_data, columns=X.columns)

prediction = model.predict(sample_df)

# Convert back to fertilizer name
result = le_fertilizer.inverse_transform(prediction)

print("Recommended Fertilizer:", result[0])

Model Accuracy: 100.0 % 
Recommended Fertilizer: DAP


In [ ]:
# Smart Soil Fertilizer Monitoring and Recommendation System (Improved)

# Ideal ranges
IDEAL = {
    "N": (50, 100),
    "P": (30, 60),
    "K": (40, 80),
    "pH": (6.0, 7.5),
    "moisture": (40, 70),   # %
    "temperature": (20, 35) # °C
}

# ---------------------------
# IoT Data Input Module
# ---------------------------
def get_sensor_data():
    print("Enter Soil Sensor Values:")

    data = {
        "N": float(input("Nitrogen (N): ")),
        "P": float(input("Phosphorus (P): ")),
        "K": float(input("Potassium (K): ")),
        "pH": float(input("Soil pH: ")),
        "moisture": float(input("Soil Moisture (%): ")),
        "temperature": float(input("Temperature (°C): ")),
        "soil_type": input("Soil Type (sandy/clay/loamy): ").lower(),
        "crop_type": input("Crop Type (rice/wheat/maize/etc): ").lower()
    }

    return data


# ---------------------------
# Smart Analysis Module
# ---------------------------
def analyze_soil(data):
    recommendations = []
    score = 0

    # Generic checker
    def check(param, value):
        low, high = IDEAL[param]

        if value < low:
            return "low"
        elif value > high:
            return "high"
        else:
            return "optimal"

    # NPK + pH analysis
    for nutrient in ["N", "P", "K"]:
        status = check(nutrient, data[nutrient])

        if status == "low":
            recommendations.append(f"Increase {nutrient} (Apply fertilizer)")
            score -= 1
        elif status == "high":
            recommendations.append(f"{nutrient} is high (Reduce usage)")
            score -= 1
        else:
            score += 1

    # pH
    ph_status = check("pH", data["pH"])
    if ph_status == "low":
        recommendations.append("Soil acidic → Add Lime")
    elif ph_status == "high":
        recommendations.append("Soil alkaline → Add Gypsum")

    # Moisture
    moisture_status = check("moisture", data["moisture"])
    if moisture_status == "low":
        recommendations.append("Low moisture → Irrigation needed")
    elif moisture_status == "high":
        recommendations.append("High moisture → Improve drainage")

    # Temperature
    temp_status = check("temperature", data["temperature"])
    if temp_status == "low":
        recommendations.append("Low temperature → Monitor crop growth")
    elif temp_status == "high":
        recommendations.append("High temperature → Increase watering")

    # Soil type logic (from paper concept)
    if data["soil_type"] == "sandy":
        recommendations.append("Sandy soil → Add organic matter (low nutrient retention)")
    elif data["soil_type"] == "clay":
        recommendations.append("Clay soil → Improve drainage")
    elif data["soil_type"] == "loamy":
        recommendations.append("Loamy soil → Ideal soil type")

    # Crop-based recommendation (basic AI idea)
    crop = data["crop_type"]
    if crop in ["rice", "wheat"]:
        recommendations.append("High nutrient crop → Ensure sufficient NPK")
    elif crop in ["pulses"]:
        recommendations.append("Legumes → Less Nitrogen needed")

    # Final decision
    if score >= 3:
        recommendations.append("Overall Soil Status: GOOD")
    else:
        recommendations.append("Overall Soil Status: NEEDS IMPROVEMENT")

    return recommendations


# ---------------------------
# Output Module
# ---------------------------
def display_results(data, recs):
    print("\n--- Soil Analysis Report ---")

    for key, value in data.items():
        print(f"{key.upper()}: {value}")

    print("\n--- Recommendations ---")
    for r in recs:
        print(f"- {r}")


# ---------------------------
# Main Function
# ---------------------------
def main():
    data = get_sensor_data()
    recs = analyze_soil(data)
    display_results(data, recs)


if __name__ == "__main__":
    main()

Enter Soil Sensor Values:
Nitrogen (N): 23
Phosphorus (P): 47
Potassium (K): 37
Soil pH: 6
Soil Moisture (%): 50
Temperature (°C): 28
Soil Type (sandy/clay/loamy): sandy
Crop Type (rice/wheat/maize/etc): maize

--- Soil Analysis Report ---
N: 23.0
P: 47.0
K: 37.0
PH: 6.0
MOISTURE: 50.0
TEMPERATURE: 28.0
SOIL_TYPE: sandy
CROP_TYPE: maize

--- Recommendations ---
- Increase N (Apply fertilizer)
- Increase K (Apply fertilizer)
- Sandy soil → Add organic matter (low nutrient retention)
- Overall Soil Status: NEEDS IMPROVEMENT


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier()


# -------------------------------
# Step 1: Load Dataset
# -------------------------------
df = pd.read_csv("dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# -------------------------------
# Step 2: Handle Missing Values
# -------------------------------
df = df.dropna()

# -------------------------------
# Step 3: Encode Categorical Data
# -------------------------------
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fertilizer = LabelEncoder()

df['Soil_Type'] = le_soil.fit_transform(df['Soil_Type'])
df['Crop_Type'] = le_crop.fit_transform(df['Crop_Type'])
df['Fertilizer_Name'] = le_fertilizer.fit_transform(df['Fertilizer_Name'])

# -------------------------------
# Step 4: Features & Target
# -------------------------------
feature_columns = ['Nitrogen', 'Phosphorous', 'Potassium', 'Temparature', 'Moisture', 'Soil_Type', 'Crop_Type']
X = df[feature_columns]
y = df['Fertilizer_Name']

# -------------------------------
# Step 5: Feature Scaling
# -------------------------------
scaler = StandardScaler()
X = scaler.fit_transform(X)

# -------------------------------
# Step 6: Train-Test Split (IMPORTANT FIX)
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,   # Changed from 0.2 to 0.3 to accommodate 3 classes
    random_state=42,
    stratify=y   # 🔥 keeps class balance
)

# -------------------------------
# Step 7: Train Model (Improved)
# -------------------------------
model = RandomForestClassifier(
    n_estimators=300,     # more trees = better learning
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------------
# Step 8: Evaluate Model
# -------------------------------
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "% ")

# -------------------------------
# Step 9: Prediction
# -------------------------------
soil_input = "Clayey"      # Corrected to 'Clayey' as per dataset
crop_input = "Wheat"

soil_encoded = le_soil.transform([soil_input])[0]
crop_encoded = le_crop.transform([crop_input])[0]

sample_data = [[30, 60, 40, 25, 60, soil_encoded, crop_encoded]]

# Convert sample to DataFrame with correct column names to avoid UserWarning
sample_df_for_scaling = pd.DataFrame(sample_data, columns=feature_columns)

# Scale input
sample_scaled = scaler.transform(sample_df_for_scaling)

prediction = model.predict(sample_scaled)
result = le_fertilizer.inverse_transform(prediction)

print("Recommended Fertilizer:", result[0])

Model Accuracy: 100.0 % 
Recommended Fertilizer: Urea


In [ ]:
# Smart Fertilizer Recommendation System (Fixed Version)

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# -------------------------------
# Step 1: Load Dataset
# -------------------------------

df = pd.read_csv("dataset.csv")

# Clean column names by replacing spaces with underscores
df.columns = df.columns.str.strip().str.replace(' ', '_')

# -------------------------------
# Step 2: Handle Missing Values
# -------------------------------

# Drop rows with missing values (simple fix)
df = df.dropna()

# -------------------------------
# Step 3: Encode Categorical Data
# -------------------------------

le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fertilizer = LabelEncoder()

df['Soil_Type'] = le_soil.fit_transform(df['Soil_Type'])
df['Crop_Type'] = le_crop.fit_transform(df['Crop_Type'])
df['Fertilizer_Name'] = le_fertilizer.fit_transform(df['Fertilizer_Name'])

# -------------------------------
# Step 4: Define Features & Target
# -------------------------------

X = df[['Nitrogen', 'Phosphorous', 'Potassium', 'Temparature', 'Moisture',
        'Soil_Type', 'Crop_Type']]

y = df['Fertilizer_Name']

# -------------------------------
# Step 5: Train-Test Split (FIXED)
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3, # Changed from 0.2 to 0.3 to prevent ValueError with stratification
    random_state=42,
    stratify=y   # 🔥 IMPORTANT FIX
)

# -------------------------------
# Step 6: Train Model
# -------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------------
# Step 7: Evaluate Model
# -------------------------------

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy: {:.2f}%".format(accuracy * 100))

# Debug check
print("\nSample Predictions vs Actual:")
print(pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
}))

# -------------------------------
# Step 8: Safe Prediction Input
# -------------------------------

# Example user input (REAL values, not encoded)
input_data = {
    'Nitrogen': 30,
    'Phosphorous': 60,
    'Potassium': 40,
    'Temparature': 25,
    'Moisture': 60,
    'Soil_Type': 'Loamy',   # change as per dataset
    'Crop_Type': 'Wheat'    # change as per dataset
}

# Convert to DataFrame
input_df = pd.DataFrame([input_data])

# Encode using SAME encoders
input_df['Soil_Type'] = le_soil.transform(input_df['Soil_Type'])
input_df['Crop_Type'] = le_crop.transform(input_df['Crop_Type'])

# Ensure correct column order
input_df = input_df[X.columns]

# Predict
prediction = model.predict(input_df)

# Decode result
fertilizer_result = le_fertilizer.inverse_transform(prediction)

print("\nRecommended Fertilizer:", fertilizer_result[0])

Model Accuracy: 100.00%

Sample Predictions vs Actual:
   Actual  Predicted
0       5          5
1       6          6
2       4          4
3       2          2
4       1          1
5       4          4
6       6          6
7       6          6
8       5          5
9       1          1

Recommended Fertilizer: Urea


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
data = pd.read_csv("dataset.csv")

# 🔥 CLEAN COLUMN NAMES
data.columns = data.columns.str.strip()

print("Columns:", data.columns)

# Encode categorical columns
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

data['Soil Type'] = le_soil.fit_transform(data['Soil Type'])
data['Crop Type'] = le_crop.fit_transform(data['Crop Type'])
data['Fertilizer Name'] = le_fert.fit_transform(data['Fertilizer Name'])

# Features (INPUT)
X = data[['Temparature', 'Humidity', 'Moisture',
          'Soil Type', 'Crop Type',
          'Nitrogen', 'Potassium', 'Phosphorous']]

# Target (OUTPUT)
y = data['Fertilizer Name']

# Train model
model = RandomForestClassifier(n_estimators=200)
model.fit(X, y)

# Accuracy (demo)
y_pred = model.predict(X)
accuracy = accuracy_score(y, y_pred)

print("✅ Accuracy:", accuracy * 100, "%")

# Save model + encoders
joblib.dump(model, "model.pkl")
joblib.dump(le_soil, "soil_encoder.pkl")
joblib.dump(le_crop, "crop_encoder.pkl")
joblib.dump(le_fert, "fert_encoder.pkl")

print("✅ Model & encoders saved")

Columns: Index(['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name'],
      dtype='object')
✅ Accuracy: 100.0 %
✅ Model & encoders saved


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
data = pd.read_csv("dataset.csv")

# 🔥 CLEAN COLUMN NAMES
data.columns = data.columns.str.strip()

print("Columns:", data.columns)

# Encode categorical columns
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

data['Soil Type'] = le_soil.fit_transform(data['Soil Type'])
data['Crop Type'] = le_crop.fit_transform(data['Crop Type'])
data['Fertilizer Name'] = le_fert.fit_transform(data['Fertilizer Name'])

# Features (INPUT)
X = data[['Temparature', 'Humidity', 'Moisture',
          'Soil Type', 'Crop Type',
          'Nitrogen', 'Potassium', 'Phosphorous']]

# Target (OUTPUT)
y = data['Fertilizer Name']

# Train model
model = RandomForestClassifier(n_estimators=200)
model.fit(X, y)

# Accuracy (demo)
y_pred = model.predict(X)
accuracy = accuracy_score(y, y_pred)

print("✅ Accuracy:", accuracy * 100, "%")

# Save model + encoders
joblib.dump(model, "model.pkl")
joblib.dump(le_soil, "soil_encoder.pkl")
joblib.dump(le_crop, "crop_encoder.pkl")
joblib.dump(le_fert, "fert_encoder.pkl")

print("✅ Model & encoders saved")

Columns: Index(['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name'],
      dtype='object')
✅ Accuracy: 100.0 %
✅ Model & encoders saved
